## MIE1624H Course Project Notebook
This notebook is part of our final course project submission. It has been cleaned to comply with submission requirements:
- API keys have been removed.
- API calls and installation steps are commented out and separated where applicable.
- All valid data processing code is retained and executable.
- Notebook is Google Colab compatible.
- Referenced data files are included in the final submission folder.

### Commented API/Installation Code (Submission Version)

In [ ]:
# (Commented out for submission)
# !pip install python-docx pymupdf nltk spacy os
# (Commented out for submission)
#     import openai
# (Commented out for submission)
#     !pip install openai
# (Commented out for submission)
#     import openai
# (Commented out for submission)
# from openai import OpenAI
# (Commented out for submission)
# client = openai.OpenAI(api_key=OPENAI_API_KEY)
# (Commented out for submission)
#     "Sweden": ["Sweden R&D investment", "Sweden AI research funding"],
# (Commented out for submission)
#     "India": ["India human capital development", "India AI research", "India startup ecosystem"],

In [ ]:
# 🔹 Install missing dependencies 
import spacy
spacy.cli.download("en_core_web_sm")

import requests
from bs4 import BeautifulSoup
from docx import Document

import json
import fitz  # PyMuPDF for PDFs
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import numpy as np
from datetime import datetime
import os

try:
except:


# 🔹 API Keys
# NEWSAPI_KEY = "your_newsapi_key_here"
# OPENAI_API_KEY = "your_openai_key_here"


# 🔹 Initialize OpenAI client (Fixed for OpenAI v1.66.3)

# 🔹 Initialize NLP tools
nltk.download("vader_lexicon")
sia = SentimentIntensityAnalyzer()

# 🔹 Expand country queries to include United Kingdom & England
COUNTRY_QUERIES = {
    "Canada": ["Canada innovation policy", "Canadian R&D funding", "Canada AI strategy"],
    "USA": ["USA unicorn startups", "US venture capital funding", "Biden tech policy"],
    "China": ["China tech adoption", "China AI regulation", "China startup growth"],
    "Germany": ["Germany industrial innovation", "Germany AI strategy", "Germany R&D spending"],
    "United Kingdom": ["UK innovation policy", "UK AI regulation", "UK startup funding"],
    "England": ["England AI strategy", "England R&D investment", "England tech startups"]
}

# 🔹 Fetch news articles, prioritizing UK sources
def fetch_news():
    articles = []
    for country, queries in COUNTRY_QUERIES.items():
        for keyword in queries:
            url = f"https://newsapi.org/v2/everything?q={keyword}&language=en&sortBy=publishedAt&apiKey={NEWSAPI_KEY}"

            if country in ["United Kingdom", "England"]:
                url += "&sources=bbc-news,the-guardian-uk,financial-times,independent,the-telegraph"

            response = requests.get(url)
            news_data = response.json()

            if news_data.get("status") != "ok":
                print(f"❌ [ERROR] Failed to fetch news for {country} - {keyword}: {news_data.get('message')}")
                continue

            for article in news_data["articles"]:
                articles.append({
                    "country": country,
                    "source": article["source"]["name"],
                    "title": article["title"],
                    "summary": article.get("description", "No summary available."),
                    "link": article["url"],
                    "published_at": article.get("publishedAt", "Unknown Date")  # ✅ Fix for missing date
                })

    print(f"✅ [DEBUG] Fetched {len(articles)} relevant innovation articles.")
    return articles[:50]  # Increased to capture more UK/England data

# 🔹 Sentiment analysis for articles
def analyze_sentiment(text):
    return sia.polarity_scores(text)["compound"]

# 🔹 Function to summarize articles using GPT-4 (Fixed API Call)
def summarize_article(content):
    if not content or content == "No summary available.":
        return "Summary unavailable due to missing content."

    response = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": f"Summarize this article, focusing on national innovation policies:\n\n{content}"}]
    )

    return response.choices[0].message.content.strip()

# 🔹 Main Execution
def main():
    articles = fetch_news()
    if not articles:
        print("❌ [ERROR] No relevant articles retrieved! Adjust query filters.")
        return

    summarized_articles = []
    for article in articles:
        summary = summarize_article(article["summary"]) if OPENAI_API_KEY else article["summary"]
        sentiment = analyze_sentiment(summary)

        summarized_articles.append({
            "country": article["country"],
            "source": article["source"],
            "title": article["title"],
            "summary": summary,
            "sentiment": sentiment,
            "link": article["link"],
            "published_at": article["published_at"]  # ✅ Fixed to avoid KeyError
        })

    # 🔹 Append detailed news aggregation results to the Appendix
    APPENDIX_SUMMARIES = {
        "Canada": [
            "AI Compute Access Fund to Boost SME Innovation",
            "Impact of Trade Policies on Canadian Tech Growth"
        ],
        "USA": [
            "Stablecoins vs CBDC — US and EU Take Different Paths",
            "US Venture Capital Growth & Startups"
        ],
        "China": [
            "China’s Global Civilization Initiative",
            "China’s AI Strategy & Tech Expansion"
        ],
        "Sweden & Germany": [
            "Copenhagen Infrastructure Partners Secures CfD Contract",
            "Germany’s AI Regulation & Innovation Funding"
        ],
        "India": [
            "ETMarkets Smart Talk",
            "India’s AI Talent & Startup Boom"
        ],
        "United Kingdom": [
            "UK Government Unveils AI Regulation Plan",
            "UK Innovation Strategy 2025 Announced"
        ],
        "England": [
            "England’s AI Regulation & Growth Policies",
            "Tech Startups in England – Challenges & Opportunities"
        ]
    }

    # 🔹 Create and structure the Word document
    doc = Document()
    doc.add_heading("Canadian Innovation Policy Review & Global Comparisons", level=1)
    doc.add_paragraph(f"Generated on {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    # 🔹 Add Summarized Articles to the Document
    doc.add_heading("Innovation Policy Summaries", level=2)
    for article in summarized_articles:
        doc.add_heading(f"{article['country']} - {article['title']}", level=3)
        doc.add_paragraph(article["summary"])
        doc.add_paragraph(f"🔹 Sentiment Score: {article['sentiment']:.2f}")
        doc.add_paragraph(f"🔹 Published Date: {article['published_at']}")
        doc.add_paragraph(f"🔹 Read more: {article['link']}")

    # 🔹 Add Appendix
    doc.add_page_break()
    doc.add_heading("Appendix A: News Aggregation Summaries", level=2)
    for country, articles in APPENDIX_SUMMARIES.items():
        doc.add_heading(f"{country} - Innovation News", level=3)
        for article_title in articles:
            doc.add_paragraph(f"🔹 {article_title}")

    # 🔹 Save the document
    report_path = "Updated_Canadian_Innovation_Strategy.docx"
    doc.save(report_path)

    print(f"✅ [SUCCESS] Saved Report: {report_path}")

# Run the script
if __name__ == "__main__":
    main()